# Intentlock: Phases 5-7: CircuitKIT, final evaluation, publishing

### Setup
Runtime → Change runtime type → **T4 GPU** (or better). Accept the Tiny Aya licence at https://huggingface.co/CohereLabs/tiny-aya-global first: the weights are gated.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo 'NO GPU: switch the runtime to a GPU'

In [ ]:
import os
if not os.path.exists('intentlock'):
    !git clone -q https://github.com/davieslennox0/intentlock.git
%cd intentlock
!pip uninstall -y -q torchao            # SafeTune's Colab note
!pip install -q -r requirements.txt 2>&1 | tail -3
!pip check | head -20

In [ ]:
from huggingface_hub import login, whoami
login()        # token with read access to Tiny Aya and write access to your namespace
HF_USER = whoami()['name']; print(HF_USER)

If this is a fresh session, pull the adapters back from the Hub.

In [ ]:
from huggingface_hub import snapshot_download
import os
for name, out in (('hardened', 'hardened-safegrad'), ('plain-sft', 'hardened-plain'), ('guard', 'guard')):
    if not os.path.exists(f'outputs/{out}/adapter'):
        snapshot_download(f'{HF_USER}/intentlock-tiny-aya-{name}', local_dir=f'outputs/{out}/adapter')

### Phase 5 (stretch): CircuitKIT discovery, then causal tests
Discovery is attribution only. The claim rests on `causal_eval`, which ablates the direction, compares against a random-direction control, and checks the held-out split. A negative verdict is a result and gets reported.

In [ ]:
!python -m interp.circuitkit_discover --n 24 --algorithm eap-ig --out reports/interp

In [ ]:
!python -m interp.causal_eval --limit 120 --out reports/interp
print(open('reports/interp/causal_results.json').read()[:3000])

### Phase 6: every configuration on both evaluation splits

In [ ]:
!python -m eval.run_all --agent-adapter outputs/hardened-safegrad/adapter --guard-adapter outputs/guard/adapter --plain-adapter outputs/hardened-plain/adapter

In [ ]:
!python -m eval.capability --out reports/capability --adapter hardened=outputs/hardened-safegrad/adapter --adapter plain_sft=outputs/hardened-plain/adapter

In [ ]:
!python -m eval.report --reports reports
print(open('reports/results_test.md').read()); print(open('reports/results_heldout_format.md').read())

Cost comparison. **Set the three prices to current public list prices**; the script records them as assumptions.

In [ ]:
# !python -m eval.cost --gpu-hourly <USD> --api-in-per-mtok <USD> --api-out-per-mtok <USD> --requests-per-day 100000

### Phase 7: publish traces, Space and collections

In [ ]:
!python scripts/publish_hf.py --user {HF_USER} --what traces space collections

In [ ]:
!cd reports && zip -qr ../reports.zip . && echo 'download reports.zip and commit reports/ to GitHub'